In [1]:
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.utils import ImageReader
import os
import io
import pandas as pd
import numpy as np
# import mne
# import json

from utils import *
from eeg_qc import compute_eeg_pipeline, test_eeg_pipeline
from ecg_qc import ecg_qc 
from eda_qc import eda_qc
from rsp_qc import *
from mic_qc import *
from lsl_problem import *
from et_qc import *
from webcam_qc import *
from behavior_qc import *
from generate_csv import *
import seaborn as sns
# import matplotlib

In [2]:
filename = 'CUNY_QC.csv'
dataset = pd.read_csv(filename)
new_df = pd.DataFrame(columns = ['Metrics', 'Mean','Standard deviation'])
new_df['Metrics'] = dataset.select_dtypes(include=['float64', 'int64']).columns.tolist()
for metric in new_df['Metrics']:
    mean_value = np.mean(dataset[metric].dropna().tolist())
    std_value = np.std(dataset[metric].dropna().tolist())
    new_df.loc[new_df['Metrics'] == metric, 'Mean'] = round(mean_value,3)
    new_df.loc[new_df['Metrics'] == metric, 'Standard deviation'] = round(std_value,3)
print(new_df)

                              Metrics      Mean Standard deviation
0             behavior_total_duration  2129.364             74.792
1   behavior_impedance_check_duration   281.364             20.781
2      behavior_average_response_time    24.273             29.809
3                    eeg_percent_good    88.093               4.45
4                    et_sampling_rate   119.996                0.0
..                                ...       ...                ...
72                duration_ps_percent    99.999              0.004
73              duration_eeg_duration  2189.566            220.981
74               duration_eeg_percent     100.0                0.0
75         duration_expected_duration  2241.017            131.033
76          duration_expected_percent     100.0                0.0

[77 rows x 3 columns]


In [ ]:
all_print_names = ['Subject',
 'Collection Date',
 'Missing stimulus markers',
 'Duration of experiment',
 'Durations do not match expected length',
 'Duration of impedence check',
 'Are all 10 seconds rest equal?',
 'Average response time across all story listening task',
 'EEG Event',
 'Percent Good before artifact removal',
 'Bad channels before robust reference',
 'Interpolated channels',
 'Bad channels after interpolation',
 'Artifactual Components Excluded',
 'ET Event',
 'Effective sampling rate',
 'Percent invalid data in left gaze point',
 'Percent invalid data in right gaze point',
 'Percent invalid data in left gaze origin',
 'Percent invalid data in right gaze origin',
 'Percent invalid data in left pupil diameter',
 'Percent invalid data in right pupil diameter',
 'Flag: all coordinates have the same % validity within each measure (LR, gaze point/origin/diameter)',
 'Flag: % of NaNs is the same between coordinate systems (UCS and TBCS (gaze origin) and between UCS and display area (gaze point))',
 'Mean difference in percent valid data between right and left eyes',
 'Percent of data with gaze point differences of over 0.2 mm',
 'ECG Event',
 'Effective sampling rate',
 'Average heart rate',
 'Kurtosis signal quality index (kSQI)',
 'Power spectrum distribution (pSQI)',
 'Relative power in baseline (basSQI)',
 'Signal to Noise Ratio',
 'EDA Event',
 'Effective sampling rate',
 'Signal integrity check',
 'Average skin conductance level',
 'Skin conductance level std',
 'Skin conductance level coefficient of variation',
 'Average amplitude of skin conductance response',
 'Skin conductance response validity',
 'Signal to noise ratio',
 'RSP Event',
 'Effective sampling rate',
 'Percent missing',
 'Signal to noise ratio',
 'Breath amplitude mean',
 'Breath amplitude std',
 'Breath amplitude min',
 'Breath amplitude max',
 'Respiration rate mean',
 'Respiration rate std',
 'Respiration rate min',
 'Respiration rate max',
 'Peak to peak interval mean',
 'Peak to peak interval std',
 'Peak to peak interval min',
 'Peak to peak interval max',
 'Baseline drift',
 'Autocorrelation at typical breath cycle',
 'Mic Event',
 'Effective sampling rate',
 'Difference between .wav file and lsl timestamps durations',
 "Number of NaN's",
 "Percent of NaN's",
 'Mic samples first quartile',
 'Mic samples third quartile',
 'Mic samples std',
 'Mic samples min',
 'Mic samples max',
 'Video Event',
 'Effective sampling rate',
 'Collected full resting state',
 'Percent of frames with face detected',
 'Number of LSL losses in ET',
 'Percentage of missing data in ET stream due to LSL',
 'Number of LSL losses in Physiological signals',
 'Percentage of missing data in Physiology stream due to LSL',
 'Number of LSL losses in Mic',
 'Percentage of missing data in Mic stream due to LSL',
 'Number of LSL losses in Video',
 'Percentage of missing data in Video stream due to LSL',
 'Number of LSL losses in EEG',
 'Percentage of missing data in EEG stream due to LSL',
 'Duration of Mic stream in seconds',
 'Duration of Mic stream in minutes and seconds',
 'Percent of Mic stream duration compared to expected duration',
 'Duration of Video stream in seconds',
 'Duration of Video stream in minutes and seconds',
 'Percent of Video stream duration compared to expected duration',
 'Duration of ET stream in seconds',
 'Duration of ET stream in minutes and seconds',
 'Percent of ET stream duration compared to expected duration',
 'Duration of Physiology stream in seconds',
 'Duration of Physiology stream in minutes and seconds',
 'Percent of Physiology stream duration compared to expected duration',
 'Duration of EEG stream in seconds',
 'Duration of EEG stream in minutes and seconds',
 'Percent of EEG stream duration compared to expected duration',
 'Expected Duration in seconds',
 'Expected Duration stream in minutes and seconds',
 'Percent of Expected Duration'
]
len(all_print_names)

#import json
#with open("metric_info.json", "r") as f:
    #data = json.load(f)
data_dict = {col: {'Description': None, 'Print Name': None, 'Unit': None} for col in dataset.columns.tolist()}
with open("data_dict.json", "w") as f:
    json.dump(data_dict, f, indent=4)
with open("data_dict.json","r") as f:
    metric_data = json.load(f)
for idx,key in enumerate(metric_data.keys()):    
    metric_data[key]['Print Name'] = all_print_names[idx]
print(metric_data)
with open("metrics_info.json","r") as f:
    metrics_info = json.load(f)
units=[]
for idx,key in enumerate(metric_data.keys()):  
    if key in list(metrics_info.keys()):
        metric_data[key]['Unit'] = metrics_info[key][1]
print(metric_data)
with open("data_dict.json", "w") as f:
    json.dump(metric_data, f, indent=4)

102

In [87]:
filename = 'CUNY_QC.csv'
dataset = pd.read_csv(filename)
quant_metrics = dataset.select_dtypes(include=['float64', 'int64']).columns.tolist()
number_of_subjects = len(dataset['Subject'])

new_df = pd.DataFrame(columns = ['Metric', 'Mean, Std. Deviation'])
new_df['Metric'] = [metric_data[metric]['Print Name'] for metric in quant_metrics]
#new_df['Metric'] = new_df['Metric'].str.wrap(30)
for metric in quant_metrics:
    mean_value = np.mean(dataset[metric].dropna().tolist())
    std_value = np.std(dataset[metric].dropna().tolist())
    #print(metric_data[metric][0], metric_data[metric][1])
    new_df.loc[new_df['Metric'] == metric_data[metric]['Print Name'], 'Mean, Std. Deviation'] = f"M = {round(mean_value,3)} {metric_data[metric]['Unit']}\nSD = {round(std_value,3)}"
    #new_df.loc[new_df['Metrics'] == metric_data[metric][0], 'Standard deviation'] = round(std_value,3)
new_df['Metric'] = new_df['Metric'].str.wrap(30)
#new_df['Mean, Std. Deviation'] = new_df['Mean, Std. Deviation'].str.wrap(12)
print(new_df)

                                               Metric  \
0                              Duration of experiment   
1                         Duration of impedence check   
2   Average response time across\nall story listen...   
3               Percent Good before artifact\nremoval   
4                             Effective sampling rate   
..                                                ...   
72  Percent of Physiology stream\nduration compare...   
73                 Duration of EEG stream in\nseconds   
74  Percent of EEG stream duration\ncompared to ex...   
75                       Expected Duration in seconds   
76                       Percent of Expected Duration   

            Mean, Std. Deviation  
0    M = 2129.364 s\nSD = 74.792  
1     M = 281.364 s\nSD = 20.781  
2      M = 24.273 s\nSD = 29.809  
3        M = 88.093 %\nSD = 4.45  
4   M = 44098.965 Hz\nSD = 0.568  
..                           ...  
72      M = 99.999 %\nSD = 0.004  
73  M = 2189.566 s\nSD = 220.981  


In [88]:
def make_distplot(metric):
    #print(metric.columns[0])
    metric = metric.rename(columns={metric.columns[0]:''})
    fig, ax = plt.subplots(figsize=(10, 2))
    sns.set_theme(style="white")
    sns.violinplot(metric, fill=False, inner='box', linewidth=2, split=True, inner_kws=dict(box_width=10, whis_width=2, color='indianred'), orient='h')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    fig.show()
    buf.seek(0)
    #plt.show()
    plt.close(fig)

    return buf

In [89]:
# PDF structure
parent_folder = os.getcwd() + '/'
pdf_path = f"{parent_folder}Dataset_Report.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=A4)
elements = []
styles = getSampleStyleSheet()
#styleN = styles["BodyText"]
styleN = styles["Normal"]
styleN.wordWrap = "CJK"


# Define subtitle style if not already done
subtitle_style = ParagraphStyle(
    name="Subtitle",
    parent=styles["Heading2"],
    fontSize=14,
    leading=16,
    textColor="gray",
    spaceAfter=12,
    alignment=1  # Centered
)

# page number function
def add_page_number(canvas, doc):
    page_num = f'{canvas.getPageNumber()}'
    canvas.setFont("Helvetica", 9)
    canvas.drawRightString(570, 20, page_num)

elements.append(Paragraph(f"Dataset Report", styles["Title"]))
elements.append(Paragraph(f"Collection Period: {dataset['Collection Date'][0].split(' ')[0]} - {dataset['Collection Date'][len(dataset['Collection Date'])-1].split(' ')[0]}, N= {number_of_subjects}", subtitle_style))
elements.append(Spacer(1, 12))

metric_data_keys = [key for key in quant_metrics]#list(metric_data.keys())
new_df_with_plots = new_df.copy()
new_df_with_plots["Distribution Plot"] = None

for idx, row in new_df_with_plots.iterrows():
    #buffer = make_distplot(dataset[['Metric']])
    buffer = make_distplot(dataset[[metric_data_keys[idx]]])
    image = ImageReader(buffer)
    orig_width, orig_height = image.getSize()
    img = Image(buffer, width=orig_width/5, height=orig_height/3)
    new_df_with_plots.at[idx, "Distribution Plot"] = img
data_list = [new_df_with_plots.columns.tolist()] + new_df_with_plots.values.tolist() 
#tableWidth = 500
#dftable = Table(data_list, colWidths=tableWidth/3, repeatRows=1)
dftable = Table(data_list, repeatRows=1)
dftable.hAlign = 'CENTER'



dftable.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.black),
    ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
    ('FONTSIZE', (0, 0), (-1, -1), 10),
    ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
]))

elements.append(dftable) 
elements.append(Spacer(1, 12))

doc.build(elements, onFirstPage = add_page_number, onLaterPages = add_page_number)
print(f'PDF created: {pdf_path}')

PDF created: /Users/apurva.gokhe/Documents/GitHub/MOBI_QC/src/MOBI_QC/important_files/Dataset_Report.pdf
